# Backbone sweep (Colab)

Fine-tunes one LLM backbone on the same radiomics prompts as the main notebook (Part A prompts, Part B QLoRA), then evaluates it. Every step is a visible cell. It is the step-by-step version of `experiments/model_sweep.py` and produces the same outputs.

**How to use**

1. Runtime → Change runtime type → **L4 GPU** (or A100 for speed).
2. Set `MODEL_NAME` and the other options in the **Configuration** cell.
3. Runtime → **Run all**.
4. To try another model: change `MODEL_NAME`, then Runtime → **Restart session** and **Run all** again. Restarting frees the GPU memory held by the previous model.
5. When finished: Runtime → **Disconnect and delete runtime**. An idle GPU still uses compute units.

Results are written to Google Drive, so they survive the session. Each run adds one row to `summary.csv`.

## 1. Setup: repository, packages, Google Drive

In [1]:
import os
if not os.path.exists("/content/thesis"):
    !git clone -q https://github.com/sezermzgl/thesis.git /content/thesis
else:
    !git -C /content/thesis pull -q
%cd /content/thesis
!pip install -q -U bitsandbytes peft accelerate mrmr-selection

from google.colab import drive
drive.mount("/content/drive")

/content/thesis
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Configuration

Candidate backbones (all have a sequence-classification head and fit in 4-bit):

| `MODEL_NAME` | size | access | suggested `BATCH` / `GRAD_ACCUM` |
|---|---|---|---|
| `Qwen/Qwen2.5-1.5B` | 1.5B | open, the current model; **run first as the control** | 8 / 2 |
| `Qwen/Qwen2.5-3B` | 3B | open | 8 / 2 |
| `Qwen/Qwen2.5-7B` | 7B | open | 4 / 4 |
| `meta-llama/Llama-2-7b-hf` | 7B | gated; the model used by Ra et al. | 4 / 4 |
| `meta-llama/Llama-3.1-8B` | 8B | gated | 4 / 4 |
| `mistralai/Mistral-7B-v0.3` | 7B | may ask to accept terms | 4 / 4 |
| `Qwen/Qwen2.5-14B`, `Qwen/Qwen3-14B-Base`, `microsoft/phi-4` | 14B | open; L4 or A100 | 2 / 8 |

For a gated model, accept its license on the Hugging Face model page with your own account, then log in in a new cell (`from huggingface_hub import login; login()`).

`BATCH × GRAD_ACCUM` = 16 in every row, so the effective batch size is the same as in the main notebook.

In [2]:
MODEL_NAME = "Qwen/Qwen2.5-7B"
N_FEATURES   = 8        # mRMR features in the prompt (main notebook: 8)
EPOCHS       = 5
LR           = 2e-4
BATCH      = 4
GRAD_ACCUM = 4
MAX_LENGTH   = 256      # raise (e.g. 512) when using more features
SEED         = 42
PRECISION    = "auto"   # "auto": bf16 on A100/L4, fp16 on T4
OUT_DIR      = "/content/drive/MyDrive/thesis_model_sweep"

# Risk levels, as in the main notebook (D6)
ALPHA_BENIGN, ALPHA_MALIGNANT, DELTA = 0.05, 0.20, 0.05

## 3. Imports, GPU and precision

In [3]:
import json, math, sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from IPython.display import display
from scipy.stats import binom
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score, recall_score, roc_auc_score
from sklearn.preprocessing import StandardScaler

REPO = Path("/content/thesis")
sys.path.insert(0, str(REPO))
import conformal_triage as ct

torch.manual_seed(SEED); np.random.seed(SEED)
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> L4 / A100 / T4"
GPU = torch.cuda.get_device_name(0)

# bf16 needs compute capability >= 8 (A100, L4). Checked directly, because
# torch.cuda.is_bf16_supported() can report emulated (slow) bf16 on a T4.
if PRECISION == "auto":
    PRECISION_USED = "bf16" if torch.cuda.get_device_capability(0)[0] >= 8 else "fp16"
else:
    PRECISION_USED = PRECISION
DTYPE = torch.bfloat16 if PRECISION_USED == "bf16" else torch.float16

RUN_TAG = f"{MODEL_NAME.split('/')[-1]}_f{N_FEATURES}_e{EPOCHS}_s{SEED}"
RUN_DIR = Path(OUT_DIR) / RUN_TAG
RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f"GPU: {GPU} | precision: {PRECISION_USED} | run: {RUN_TAG}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GiB")

GPU: NVIDIA L4 | precision: bf16 | run: Qwen2.5-7B_f8_e5_s42
VRAM: 22.0 GiB


## 4. Data: cached radiomics tables

In [4]:
SPLITS = ["train", "validation", "calibration", "test"]
tables = {s: pd.read_csv(REPO / "features" / f"radiomics_{s}.csv") for s in SPLITS}

display(pd.DataFrame({s: {"nodules": len(t), "malignant": int(t["label"].sum()),
                          "malignant share": round(t["label"].mean(), 3)}
                      for s, t in tables.items()}).T)

,nodules,malignant,malignant share
train,2015.0,682.0,0.338
validation,432.0,146.0,0.338
calibration,432.0,146.0,0.338
test,614.0,236.0,0.384


## 5. Train-only standardization and mRMR feature selection

Same as main notebook A10: median imputation and z-scoring are fitted on train only, and mRMR picks `N_FEATURES` features on train.

In [5]:
from mrmr import mrmr_classif

META_COLUMNS = ["id", "sample_id", "image_name", "official_split", "analysis_split",
                "label", "label_name", "mask_source", "source_split"]
train = tables["train"]
feature_columns = [c for c in train.columns
                   if c not in META_COLUMNS and pd.api.types.is_numeric_dtype(train[c])]
raw = {s: t.reindex(columns=feature_columns).replace([np.inf, -np.inf], np.nan) for s, t in tables.items()}
valid = [c for c in feature_columns
         if raw["train"][c].notna().sum() > 0 and raw["train"][c].nunique(dropna=True) > 1]

imputer, scaler = SimpleImputer(strategy="median"), StandardScaler()
scaled = {"train": pd.DataFrame(scaler.fit_transform(imputer.fit_transform(raw["train"][valid])), columns=valid)}
for s in SPLITS[1:]:
    scaled[s] = pd.DataFrame(scaler.transform(imputer.transform(raw[s][valid])), columns=valid)

selected = list(mrmr_classif(X=scaled["train"], y=pd.Series(train["label"].astype(int)),
                             K=min(N_FEATURES, len(valid)), show_progress=False))
print(f"{len(valid)} usable features; mRMR selected {len(selected)}:")
for i, f in enumerate(selected, 1):
    print(f"  {i:2d}. {f}")

67 usable features; mRMR selected 8:
   1. original_shape2D_MajorAxisLength
   2. original_glszm_HighGrayLevelZoneEmphasis
   3. original_shape2D_Elongation
   4. original_glcm_ClusterShade
   5. original_firstorder_Range
   6. original_glszm_LargeAreaHighGrayLevelEmphasis
   7. original_shape2D_MaximumDiameter
   8. original_firstorder_Maximum


## 6. Prompts (identical format to the main notebook)

In [6]:
data = {}
for s in SPLITS:
    X = scaled[s][selected].to_numpy(dtype=np.float32)
    meta = tables[s][["id", "official_split", "label"]].reset_index(drop=True)
    data[s] = pd.DataFrame({
        "sample_id": meta["official_split"].astype(str) + "/" + meta["id"].astype(str),
        "label": meta["label"].astype(int),
        "prompt": [ct.render_prompt(selected, x) for x in X],
    })

print("Example prompt (validation, first nodule):\n")
print(data["validation"]["prompt"].iloc[0])

Example prompt (validation, first nodule):

Classify this thyroid nodule as benign or malignant based on the following standardized radiomic features. The 'Major Axis Length' feature is measured at -1.140. The 'High Gray Level Zone Emphasis' feature is measured at -0.552. The 'Elongation' feature is measured at 0.634. The 'Cluster Shade' feature is measured at -0.465. The 'Range' feature is measured at -1.113. The 'Large Area High Gray Level Emphasis' feature is measured at -0.645. The 'Maximum Diameter' feature is measured at -1.155. The 'Maximum' feature is measured at -1.325.


## 7. Load the backbone in 4-bit and add LoRA

LoRA rank 16 on the attention Q/V projections, as in the main notebook. Architectures with a fused QKV projection (e.g. Phi) use `qkv_proj`.

In [7]:
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForSequenceClassification, AutoTokenizer, BitsAndBytesConfig

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2, device_map="auto", torch_dtype=DTYPE,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=DTYPE))
model.config.pad_token_id = tokenizer.pad_token_id
model = prepare_model_for_kbit_training(model)

module_names = {n.split(".")[-1] for n, _ in model.named_modules()}
TARGETS = ["q_proj", "v_proj"] if {"q_proj", "v_proj"} <= module_names else ["qkv_proj"]
model = get_peft_model(model, LoraConfig(
    task_type=TaskType.SEQ_CLS, r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    target_modules=TARGETS))
print("LoRA targets:", TARGETS)
model.print_trainable_parameters()
print(f"VRAM allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GiB")

config.json:   0%|          | 0.00/686 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[transformers] Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-7B
Key            | Status     | 
---------------+------------+-
lm_head.weight | UNEXPECTED | 
score.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


LoRA targets: ['q_proj', 'v_proj']
trainable params: 5,053,440 || all params: 7,075,679,744 || trainable%: 0.0714
VRAM allocated: 5.20 GiB


## 8. Tokenized datasets

In [8]:
from torch.utils.data import Dataset

class PromptDataset(Dataset):
    def __init__(self, df):
        self.texts, self.labels = df["prompt"].tolist(), df["label"].tolist()
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        enc = tokenizer(self.texts[i], truncation=True, max_length=MAX_LENGTH,
                        padding="max_length", return_tensors="pt")
        return {"input_ids": enc["input_ids"].squeeze(0),
                "attention_mask": enc["attention_mask"].squeeze(0),
                "labels": torch.tensor(self.labels[i], dtype=torch.long)}

longest = max(len(tokenizer(p)["input_ids"]) for s in SPLITS for p in data[s]["prompt"])
print(f"Longest prompt: {longest} tokens (MAX_LENGTH = {MAX_LENGTH})")
assert longest <= MAX_LENGTH, f"Prompts would be truncated: set MAX_LENGTH above {longest}"
train_ds, val_ds = PromptDataset(data["train"]), PromptDataset(data["validation"])

Longest prompt: 157 tokens (MAX_LENGTH = 256)


## 9. Fine-tune

Class-weighted cross-entropy, cosine schedule with 5% warmup, the best epoch chosen by validation AUC. `model_accepts_loss_kwargs = False` makes the Trainer divide the loss by the accumulation steps (the gradient-accumulation fix from main notebook B5). The table below updates after each epoch.

In [9]:
from transformers import Trainer, TrainingArguments

counts = data["train"]["label"].value_counts().sort_index()
class_weights = torch.tensor((counts.sum() / (2.0 * counts)).values, dtype=torch.float32)
print("class weights [benign, malignant]:", class_weights.tolist())

class WeightedTrainer(Trainer):
    def __init__(self, *a, **kw):
        super().__init__(*a, **kw)
        self.model_accepts_loss_kwargs = False

    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop("labels")
        out = model(**inputs)
        loss = nn.CrossEntropyLoss(weight=class_weights.to(out.logits.device))(out.logits, labels)
        return (loss, out) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    p = torch.softmax(torch.tensor(np.asarray(logits, dtype=np.float32)), dim=-1).numpy()[:, 1]
    pred = (p >= 0.5).astype(int)
    return {"auc": roc_auc_score(labels, p), "acc": accuracy_score(labels, pred),
            "sensitivity": recall_score(labels, pred)}

steps_per_epoch = math.ceil(len(train_ds) / (BATCH * GRAD_ACCUM))
trainer = WeightedTrainer(
    model=model,
    args=TrainingArguments(
        output_dir=str(RUN_DIR / "checkpoints"), num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH, per_device_eval_batch_size=BATCH * 2,
        gradient_accumulation_steps=GRAD_ACCUM, learning_rate=LR,
        lr_scheduler_type="cosine", warmup_steps=math.ceil(0.05 * steps_per_epoch * EPOCHS),
        logging_steps=20, save_strategy="epoch", eval_strategy="epoch", save_total_limit=1,
        load_best_model_at_end=True, metric_for_best_model="auc",
        bf16=PRECISION_USED == "bf16", fp16=PRECISION_USED == "fp16",
        gradient_checkpointing=True, report_to="none", seed=SEED),
    train_dataset=train_ds, eval_dataset=val_ds, compute_metrics=compute_metrics)

start = time.time()
trainer.train()
TRAIN_MINUTES = round((time.time() - start) / 60, 1)
trainer.save_model(str(RUN_DIR / "adapter"))
pd.DataFrame(trainer.state.log_history).to_csv(RUN_DIR / "train_log.csv", index=False)
print(f"Training took {TRAIN_MINUTES} min; adapter saved to {RUN_DIR / 'adapter'}")

class weights [benign, malignant]: [0.7558139562606812, 1.4772727489471436]


Epoch,Training Loss,Validation Loss,Auc,Acc,Sensitivity
1,0.760953,0.775367,0.673664,0.449074,0.924658
2,0.785452,0.729860,0.680764,0.557870,0.849315
3,0.723550,0.874358,0.697002,0.527778,0.856164
4,0.638513,0.613711,0.685303,0.675926,0.472603
5,0.575250,0.644277,0.692966,0.634259,0.671233


Training took 57.3 min; adapter saved to /content/drive/MyDrive/thesis_model_sweep/Qwen2.5-7B_f8_e5_s42/adapter


## 10. Validation metrics per epoch

In [10]:
log = pd.DataFrame(trainer.state.log_history)
per_epoch = log.dropna(subset=["eval_auc"])[["epoch", "eval_loss", "eval_auc", "eval_acc", "eval_sensitivity"]]
display(per_epoch.round(4).reset_index(drop=True))
print("Best validation AUC:", round(per_epoch["eval_auc"].max(), 4))

,epoch,eval_loss,eval_auc,eval_acc,eval_sensitivity
0,1.0,0.7754,0.6737,0.4491,0.9247
1,2.0,0.7299,0.6808,0.5579,0.8493
2,3.0,0.8744,0.6970,0.5278,0.8562
3,4.0,0.6137,0.6853,0.6759,0.4726
4,5.0,0.6443,0.6930,0.6343,0.6712


Best validation AUC: 0.697


## 11. Predict P(malignant) on validation, calibration and test

In [11]:
model = trainer.model
probs, labels = {}, {}
for s in ["validation", "calibration", "test"]:
    probs[s] = ct.predict_malignant_proba(model, tokenizer, data[s]["prompt"].tolist(),
                                          batch_size=BATCH * 2, max_length=MAX_LENGTH)
    labels[s] = data[s]["label"].to_numpy()
    data[s].assign(prob_malignant=probs[s]).drop(columns="prompt").to_csv(RUN_DIR / f"probs_{s}.csv", index=False)
print("Saved per-nodule probabilities to", RUN_DIR)

Saved per-nodule probabilities to /content/drive/MyDrive/thesis_model_sweep/Qwen2.5-7B_f8_e5_s42


## 12. Classification metrics

`auc_correct_softmax` measures how well softmax confidence (distance of P from 0.5) separates correct from incorrect predictions; 0.5 is chance. This is report Section 10.2.

In [12]:
cls = {}
for s in ["validation", "calibration", "test"]:
    p, y = probs[s], labels[s]
    pred = (p >= 0.5).astype(int)
    cls[s] = {"acc": accuracy_score(y, pred), "f1": f1_score(y, pred), "auc": roc_auc_score(y, p),
              "sens": recall_score(y, pred), "auc_correct_softmax": roc_auc_score(pred == y, np.abs(p - 0.5))}
cls = pd.DataFrame(cls).T
display(cls.round(3))

,acc,f1,auc,sens,auc_correct_softmax
validation,0.528,0.551,0.697,0.856,0.481
calibration,0.542,0.564,0.686,0.877,0.448
test,0.581,0.586,0.669,0.771,0.500


## 13. LTT on the softmax score (calibration → test)

Same rule and candidate thresholds as main notebook D6: P ≤ t is auto-benign (R1: share of cancers ≤ `ALPHA_BENIGN`), P ≥ u is auto-malignant (R2: share of benign ≤ `ALPHA_MALIGNANT`). Thresholds are tested from the safest candidate onwards, stopping at the first failure. `certified = False` everywhere means nothing could be guaranteed.

In [13]:
P_T_GRID = [round(x, 2) for x in np.arange(0.05, 0.50, 0.05)]
P_U_GRID = [round(x, 2) for x in np.arange(0.95, 0.50, -0.05)]
pc, yc, pt, yt = probs["calibration"], labels["calibration"], probs["test"], labels["test"]

rule = ct.calibrate_triage(pc, yc, P_T_GRID, P_U_GRID, ALPHA_BENIGN, ALPHA_MALIGNANT, DELTA)
print(f"R1 (auto-benign), alpha = {ALPHA_BENIGN}:"); display(rule.benign_table.round(4))
print(f"R2 (auto-malignant), alpha = {ALPHA_MALIGNANT}:"); display(rule.malignant_table.round(4))
print(f"Locked rule: t = {rule.t}, u = {rule.u}")
display(ct.triage_report(rule.decide(pt), yt).to_frame("test").round(3))

R1 (auto-benign), alpha = 0.05:


,threshold,n,errors,observed_rate,p_value,certified
0,0.05,0,0,NaN,1.0000,False
1,0.10,0,0,NaN,1.0000,False
2,0.15,0,0,NaN,1.0000,False
3,0.20,0,0,NaN,1.0000,False
4,0.25,0,0,NaN,1.0000,False
5,0.30,14,4,0.2857,0.9996,False
6,0.35,50,6,0.1200,0.9882,False
7,0.40,88,14,0.1591,1.0000,False
8,0.45,116,17,0.1466,1.0000,False


R2 (auto-malignant), alpha = 0.2:


,threshold,n,errors,observed_rate,p_value,certified
0,0.95,0,0,NaN,1.0000,False
1,0.90,63,24,0.3810,0.9997,False
2,0.85,98,39,0.3980,1.0000,False
3,0.80,123,56,0.4553,1.0000,False
4,0.75,165,89,0.5394,1.0000,False
5,0.70,217,119,0.5484,1.0000,False
6,0.65,250,144,0.5760,1.0000,False
7,0.60,275,155,0.5636,1.0000,False
8,0.55,297,170,0.5724,1.0000,False


Locked rule: t = None, u = None


,test
n,614.0
auto_benign,0.0
auto_benign_malignant,0.0
auto_benign_malignant_rate,NaN
auto_malignant,0.0
auto_malignant_benign,0.0
auto_malignant_benign_rate,NaN
referred,614.0
automation_rate,0.0
cancers_auto_benign_share,0.0


## 14. Miss-rate guarantee (exploratory, report Section 10.3)

Bounds the share of **all** cancers sent to auto-benign (P ≤ t). Candidates are fixed in advance (0.01 to 0.50). Calibration and test are not exchangeable (report Section 5), so the realised test miss rate can exceed the target.

In [14]:
MISS_GRID = [round(x, 2) for x in np.arange(0.01, 0.51, 0.01)]
n_cancer = int((yc == 1).sum())

miss = {}
for alpha in (0.05, 0.10):
    t = None
    for cand in MISS_GRID:
        k = int(((pc <= cand) & (yc == 1)).sum())
        if binom.cdf(k, n_cancer, alpha) > DELTA:
            break
        t = cand
    miss[f"miss rate <= {alpha:.0%}"] = {
        "threshold t": t,
        "calibration automation": (pc <= t).mean() if t is not None else np.nan,
        "test automation": (pt <= t).mean() if t is not None else np.nan,
        "test miss rate": ((pt <= t) & (yt == 1)).sum() / (yt == 1).sum() if t is not None else np.nan,
    }
miss = pd.DataFrame(miss).T
display(miss.round(3))

,threshold t,calibration automation,test automation,test miss rate
miss rate <= 5%,0.28,0.023,0.060,0.030
miss rate <= 10%,0.36,0.141,0.228,0.123


## 15. Save the run and compare with earlier runs

In [15]:
row = {"run": RUN_TAG, "model": MODEL_NAME, "n_features": len(selected), "epochs": EPOCHS,
       "gpu": GPU, "precision": PRECISION_USED, "minutes": TRAIN_MINUTES}
for s in cls.index:
    for m in cls.columns:
        row[f"{s}_{m}"] = cls.loc[s, m]
row.update({"ltt_t": rule.t, "ltt_u": rule.u,
            "ltt_test_auto_benign": int((rule.decide(pt) == 0).sum()),
            "ltt_test_auto_malignant": int((rule.decide(pt) == 1).sum())})
for alpha, tag in ((0.05, "miss5"), (0.10, "miss10")):
    r = miss.loc[f"miss rate <= {alpha:.0%}"]
    row.update({f"{tag}_t": r["threshold t"], f"{tag}_cal_automation": r["calibration automation"],
                f"{tag}_test_automation": r["test automation"], f"{tag}_test_miss_rate": r["test miss rate"]})

(RUN_DIR / "config.json").write_text(json.dumps(
    {"model": MODEL_NAME, "n_features": N_FEATURES, "epochs": EPOCHS, "lr": LR, "batch": BATCH,
     "grad_accum": GRAD_ACCUM, "max_length": MAX_LENGTH, "seed": SEED, "precision": PRECISION_USED,
     "gpu": GPU, "lora_targets": TARGETS, "selected_features": selected}, indent=2))
pd.Series(row).to_json(RUN_DIR / "metrics.json", indent=2)
summary_path = Path(OUT_DIR) / "summary.csv"
pd.DataFrame([row]).to_csv(summary_path, mode="a", header=not summary_path.exists(), index=False)

summary = pd.read_csv(summary_path)
cols = ["run", "gpu", "precision", "minutes", "validation_auc", "calibration_auc", "test_auc",
        "calibration_auc_correct_softmax", "ltt_t", "ltt_u", "miss5_t", "miss5_test_miss_rate"]
print("All runs so far:")
display(summary[[c for c in cols if c in summary.columns]].round(3))
print("Saved:", RUN_DIR, "and", summary_path)

All runs so far:


,run,gpu,precision,minutes,validation_auc,calibration_auc,test_auc,calibration_auc_correct_softmax,ltt_t,ltt_u,miss5_t,miss5_test_miss_rate
0,Qwen2.5-1.5B_f8_e5_s42,NVIDIA L4,bf16,16.2,0.697,0.714,0.667,0.658,NaN,NaN,0.22,0.081
1,Qwen2.5-7B_f8_e5_s42,NVIDIA L4,bf16,57.3,0.697,0.686,0.669,0.448,NaN,NaN,0.28,0.030


Saved: /content/drive/MyDrive/thesis_model_sweep/Qwen2.5-7B_f8_e5_s42 and /content/drive/MyDrive/thesis_model_sweep/summary.csv


## Done

- Results are in `OUT_DIR` on Google Drive. Send `summary.csv` (and the run folders if needed) for the comparison.
- Next model: change `MODEL_NAME` in the Configuration cell, then **Restart session** and **Run all**.
- Finished for now: Runtime → **Disconnect and delete runtime**.

In [16]:
!cd /content/drive/MyDrive && zip -qr /content/sweep_results.zip thesis_model_sweep -x "*/adapter/*" "*/checkpoints/*"
from google.colab import files; files.download("/content/sweep_results.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>